In [2]:
import pandas as pd

df = pd.read_csv("college_qb_stats.csv")

if "Rate.1" in df.columns:
    df = df.drop(columns=["Rate.1"])

if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])

df = df.loc[:, ~df.columns.str.contains("Unnamed")]

# columns getting qb prefix
qb_stat_cols = [
    "Cmp", "Att", "Inc", "Cmp%", "Yds", "TD", "Int",
    "TD%", "Int%", "Y/A", "AY/A", "Y/C", "Y/G"
]

# renaming only those columns
rename_dict = {col: f"qb_{col}" for col in qb_stat_cols if col in df.columns}
df = df.rename(columns=rename_dict)

df = df.reset_index(drop=True)

df.to_csv("college_qb_stats_clean.csv", index=False)

In [54]:
df = pd.read_csv("college_rb_stats1.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

# wrong column
df = df.drop(df.columns[3], axis=1)

rename_dict = {
    # rushing
    "Y/G": "rb_rush_Y/G",
    "Att": "rb_rush_Att",
    "Yds": "rb_rush_Yds",
    "Y/A": "rb_rush_Y/A",
    "TD": "rb_rush_TD",

    # receiving
    "Rec": "rb_rec_Rec",
    "Yds.1": "rb_rec_Yds",
    "Y/R": "rb_rec_Y/R",
    "TD.1": "rb_rec_TD",
    "Y/G.2": "rb_rec_Y/G",
}

df = df.rename(columns=rename_dict)

df = df.rename(columns={"rb_rec_Yds": "rb_rush_Yds"})

# removing extras
df = df.drop(columns=[c for c in ["Yds.2", "Y/G.1", "TD.2"] if c in df.columns])

df = df.reset_index(drop=True)

df.to_csv("college_rb_stats_clean1.csv", index=False)

In [57]:
df = pd.read_csv("college_wrte_stats1.csv")

if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])
df = df.loc[:, ~df.columns.str.contains("Unnamed")]

if "Y/G.1" in df.columns:
    df = df.drop(columns=["Y/G.1"])

receiving_cols = {
    "Rec": "rec_Rec",
    "Yds": "rec_Yds",
    "Y/R": "rec_Y/R",
    "TD": "rec_TD",
    "Y/G": "rec_Y/G"
}

rename_dict = {col: new_col for col, new_col in receiving_cols.items() if col in df.columns}
df = df.rename(columns=rename_dict)

df = df.reset_index(drop=True)

df.to_csv("college_wrte_stats_clean1.csv", index=False)

In [60]:
df = pd.read_csv("college_ol_stats1.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]  # remove Unnamed
if "-9999" in df.columns:
    df = df.drop(columns=["-9999"])
if "Player-additional" in df.columns:
    df = df.drop(columns=["Player-additional"])

cols_to_drop = []
if "G.1" in df.columns:
    cols_to_drop.append("G.1")
if "Yds.1" in df.columns:
    cols_to_drop.append("Yds.1")
df = df.drop(columns=cols_to_drop)

df = df.reset_index(drop=True)

df.to_csv("college_ol_stats_clean1.csv", index=False)

In [61]:
df = pd.read_csv("college_dllb_stats1.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

df = df.reset_index(drop=True)

redundant_sk_col = df.columns[2]  
df = df.drop(columns=[redundant_sk_col])

if "Sk.1" in df.columns:
    df = df.rename(columns={"Sk.1": "Sk"})

df["Sk"] = df["Sk"].astype(int)

# adding sacks/game
df["Sk/G"] = df["Sk"] / df["G"]

for col in ["Yds", "Yds.1"]:
    if col in df.columns:
        df = df.drop(columns=[col])
        
df.to_csv("college_dllb_stats_clean1.csv", index=False)

In [63]:
df = pd.read_csv("college_db_stats1.csv")

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df = df.drop(columns=["-9999"], errors="ignore")

redundant_int_col = df.columns[2]  
df = df.drop(columns=[redundant_int_col])

if "Int.1" in df.columns:
    df = df.rename(columns={"Int.1": "Int"})

columns_to_drop = ["Yds", "Yds.1", "Yds.2", "TotOff", "Touch"]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df = df.reset_index(drop=True)

df.to_csv("college_db_stats_clean1.csv", index=False)

In [64]:
qb = pd.read_csv("college_qb_stats_clean.csv")
rb = pd.read_csv("college_rb_stats_clean1.csv")
wrte = pd.read_csv("college_wrte_stats_clean1.csv")
ol = pd.read_csv("college_ol_stats_clean1.csv")
dllb = pd.read_csv("college_dllb_stats_clean1.csv")
db = pd.read_csv("college_db_stats_clean1.csv")

dfs = [qb, rb, wrte, ol, dllb, db]

combined_df = pd.concat(dfs, ignore_index=True, sort=False)

if "Rk" in combined_df.columns:
    combined_df = combined_df.drop(columns=["Rk"])

# sorting by draft yr
combined_df = combined_df.reset_index(drop=True)
combined_df = combined_df.sort_values(by=["Draft Year", "Round", "Pick"], ignore_index=True)

combined_df.to_csv("college_all_positions_clean.csv", index=False)

In [65]:
combined_df

,Player,Rate,Draft Team,Round,Pick,Draft Year,Draft College,From,To,G,...,rec_TD,Sk,Solo,Ast,Comb,TFL,Sk/G,Int,IntTD,PD
0,Cam Newton,178.2,CAR,1,1,2011,Auburn,2007,2010,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Von Miller,NaN,DEN,1,2,2011,Texas A&M,2007,2010,47.0,...,NaN,33.0,104.0,77.0,181.0,50.5,0.702128,NaN,NaN,NaN
2,Marcell Dareus,NaN,BUF,1,3,2011,Alabama,2009,2010,25.0,...,NaN,11.0,39.0,27.0,66.0,20.0,0.440000,NaN,NaN,NaN
3,A.J. Green,NaN,CIN,1,4,2011,Georgia,2008,2010,32.0,...,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Patrick Peterson,NaN,CRD,1,5,2011,LSU,2008,2010,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2316,Brian Cole,NaN,MIN,7,249,2020,Mississippi State,2018,2019,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,2.0
2317,Tremayne Anchrum,NaN,RAM,7,250,2020,Clemson,2017,2017,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2318,Stephen Sullivan,NaN,SEA,7,251,2020,LSU,2017,2019,41.0,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2319,Tyrie Cleveland,NaN,DEN,7,252,2020,Florida,2016,2019,46.0,...,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
